# Validation Test - Static droplet on slip wall (transient simulation)

This notebook and the corresponding evaluation notebook (StaticDropletOnSlipWall_transient_Evaluation.ipynb) are part of published results (cf. 5.1.3) found in *The extended discontinuous Galerkin method adapted for moving contact line problems via the generalized Navier boundary condition* (https://doi.org/10.1002/fld.5016).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
ExecutionQueues

Error: (1,15): error CS0103: The name 'ExecutionQueues' does not exist in the current context

In [3]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [4]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [5]:
wmg.Init("CLpaper_StaticDropletOnSlipWall", myBatch);

In [6]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [7]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [20]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

In [ ]:
// foreach (var s in sessions) {
//      if(s.Name.Contains("StaticDropletOnWall_transient") && s.CreationTime.Date == new DateTime(2025, 11, 25)) {
//         s.Delete(true);
//         //Console.WriteLine($"Deleted session from {s.CreationTime.Date}");
//     }
// }

In [ ]:
// static var sessions = BoSSSshell.WorkflowMgm.Sessions;
// sessions

## Grid Creation for Mesh Study

In [ ]:
int[] Resolutions = new int[] { 20, 40, 60 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

double L = 0.5; 

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"StatitcDropletOnWall_{Res}";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = GenericBlas.Linspace(-L, L, Res + 1);
        double[] yNodes = GenericBlas.Linspace( 0, L, (Res / 2) + 1);
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);
    
        grd.Name = GridName;
    
        grd.DefineEdgeTags(delegate (double[] X) {
            string ret = null;
            if (Math.Abs(X[0] + L) <= 1.0e-8)
                ret = IncompressibleBcType.NavierSlip_Linear.ToString();
            if (Math.Abs(X[0] - L) <= 1.0e-8)
                ret = IncompressibleBcType.NavierSlip_Linear.ToString();
            if (Math.Abs(X[1]) <= 1.0e-8)
                ret = IncompressibleBcType.NavierSlip_Linear.ToString();
            if (Math.Abs(X[1] - L) <= 1.0e-8)
                ret = IncompressibleBcType.NavierSlip_Linear.ToString();
            return ret;
        });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

## Initial Values

In [10]:
double r = 0.25;
r

0.25

In [11]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double dropRadius = 0.25;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2()) - dropRadius; " + 
    "}");

## Physical Settings

In [12]:
double rho = 1e4;    
double mu = 1.0;
double sigma = 1.0;

double La = sigma * rho * 2 * r / mu.Pow2();
La

5000

In [27]:
double dt = 0.01;
static double t_end = 125;

## Control settings

In [14]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
for(int iGrd = 0; iGrd < Grids.Length; iGrd++) {
    
    XNSE_Control C = new XNSE_Control();
    
    int pDeg = 2;   
    var grd  = Grids[iGrd];

    C.SetDGdegree(pDeg);
    
    // set grid
    C.SetGrid(grd);
    
    // initial conditions   
    C.AddInitialValue("Phi", PhiFunc);   
    
    // C.PostprocessingModules.Add(new Dropletlike(){ SolverStage = 2, LogPeriod = 10});
    // C.PostprocessingModules.Add(new EnergyLogging(){ SolverStage = 2, LogPeriod = 10});

    C.PhysicalParameters.rho_A = rho;
    C.PhysicalParameters.rho_B = rho;
    C.PhysicalParameters.mu_A  = mu;
    C.PhysicalParameters.mu_B  = mu;
    C.PhysicalParameters.Sigma = sigma;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;

    C.CutCellQuadratureType = CutCellQuadratureMethod.Saye;

    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.PhysicalParameters.betaL = 0.0;
    C.PhysicalParameters.betaS_A = 2000.0;
    C.PhysicalParameters.betaS_B = 2000.0;
    C.PhysicalParameters.theta_e = Math.PI / 2.0;
    
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.NonLinearSolver.MaxSolverIterations = 50;
    
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
  
    C.dtMin         = dt;
    C.dtMax         = dt;
    C.Endtime       = t_end;
    C.NoOfTimesteps = (int)(t_end/dt); 
    
    C.saveperiod = 125;
    
    C.SessionName = "StaticDropletOnWall_transient_meshStudy_mesh" + Resolutions[iGrd];
    
    Controls.Add(C);
}

In [16]:
foreach (var ctrl in Controls) {
    Console.WriteLine($"{ctrl.SessionName}");
    // Console.WriteLine($"{ctrl.SessionName}: dt = {ctrl.dtMin}; saveperiod = {ctrl.saveperiod}; logperiod = {ctrl.PostprocessingModules.Pick(0).LogPeriod}");
}

In [17]:
int NoCtrls = Controls.Count();
NoCtrls

3

## Launch/Restart Jobs

In [ ]:
public static Job CreateRestartJob(Job job2rest) {

    string baseName = job2rest.LatestSession.Name;

    // find latest session with same base name (in case of multiple restarts)
    var studySess = sessions.Where(sess => sess.Name.Contains(baseName));
    int studyCount = studySess.Count();
    studySess = studySess.OrderBy(sess => sess.CreationTime);
    var LatestSession = studySess.Last();
    if (LatestSession.SuccessfulTermination) {
        Console.WriteLine($"Latest session {LatestSession.Name} already terminated successfully; no restart job created.");
        return LatestSession.GetControl().CreateJob();
    }
    Console.WriteLine($"Creating restart job for session {LatestSession.Name} ...");

    var rCtrl = LatestSession.GetControl().CloneAs();
    // var rCtrl = job2rest.LatestSession.GetControl().CloneAs();

    Guid rst_ID = LatestSession.ID;
    TimestepNumber rst_ts = LatestSession.Timesteps.Last().TimeStepNumber;

    rCtrl.InitialValues.Clear();
    rCtrl.InitialValues_Evaluators.Clear();

    rCtrl.Endtime = t_end;
    rCtrl.RestartInfo = Tuple.Create(rst_ID, rst_ts);

    rCtrl.SessionName = baseName + "_Restart" + studyCount;

    var rJob = rCtrl.CreateJob();

    // set resources
    rJob.NumberOfMPIProcs = job2rest.NumberOfMPIProcs;

    int numThreads = job2rest.NumberOfThreads;
    rJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = rJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    rJob.MySetCommandLineArguments(args.Values.ToArray());

    rJob.RetryCount = job2rest.RetryCount;

    Console.WriteLine($"Restart job for session {rCtrl.SessionName} created.");

    return rJob;
}

In [26]:
List<Job> jobs = new List<Job>();

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);
    
    if (oneJob.Status == JobStatus.FailedOrCanceled) {
        Console.WriteLine($"Job for session {ctrl.SessionName} failed. Trying to restart latest job ...");

        var restartJob = CreateRestartJob(oneJob);
        restartJob.Activate(myBatch);
        jobs.Add(restartJob);

    } else {   
          
        //oneJob.Activate(myBatch);
        jobs.Add(oneJob);
    }
}

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletInEquilibrium_meshStudy_mesh20 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\XNSEpaper_DropletInEquilibrium-XNSE_Solver2025Oct13_132951.571695
copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletInEquilibrium_meshStudy_mesh40 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\XNSEpaper_DropletInEquilibrium-XNSE_Solver2025Oct13_133001.521669
copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.


## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress 
                                    || job.Status == JobStatus.PendingInExecutionQueue
                                    || job.Status == JobStatus.PreActivation).Count();
NoInProgress

In [ ]:
int maxDays = 7;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [ ]:
int NoFailed = jobs.Where(job => job.Status == JobStatus.FailedOrCanceled).Count();
NoFailed

In [ ]:
int NoSuccess = jobs.Where(job => job.Status == JobStatus.FinishedSuccessful).Count();
NoSuccess

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var ctrl in Controls) {
        var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(ctrl.SessionName));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {ctrl.SessionName} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {ctrl.SessionName}!"); // should not happen
            } 
        }
    }

}

In [ ]:
NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");

Error: (1,37): error CS0103: The name 'NoSuccess' does not exist in the current context